# Protein Pretrain (Unified)

This notebook replaces the separate from-scratch, resume, and streaming notebooks.
It loads `config/train.resume.yaml` by default for controlled continuation, or `config/train.yaml` / `config/train.16gb.yaml` for fresh training.

Supports:
- **Windows local/Jupyter**: notebook bootstrap forces safe worker settings when needed.
- **Ubuntu/Linux/cloud VM**: same config, CUDA auto-detection, streaming train parts.
- **Google Colab**: set `MDNAC_REPO_DIR` or clone/mount the repo under `/content` or Google Drive.
- **train_from_scratch**, **resume**, and **auto** modes.

Default config: `config/train.resume.yaml`. Use `MDNAC_TRAIN_CONFIG` only when you want to point at another YAML. Use `MDNAC_REPO_DIR` only when the notebook is not opened from inside the cloned repo.

Data source:
- Local `train.txt` or `train_part_*.txt` files only

After training, the model checkpoint can be uploaded to **HuggingFace** if `HF_TOKEN` is set.

> Put `train_part_*.txt` files in `data/cache/protein_train_parts/`, or provide `data/compiled/refseq_bacteria_protein/train.txt`.


In [1]:
import os
import sys
from pathlib import Path

def find_repo_dir_for_import(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    repo_dir_env = os.environ.get("MDNAC_REPO_DIR")
    if repo_dir_env:
        candidates.append(Path(repo_dir_env).expanduser())
    candidates.extend([
        Path("/content/MDNAC"),
        Path("/content/drive/MyDrive/MDNAC"),
    ])
    for candidate in candidates:
        resolved = candidate.expanduser().resolve()
        if (resolved / "pyproject.toml").exists() and (resolved / "libs").is_dir():
            return resolved
    raise RuntimeError(
        "Could not locate repo. Run inside the repo, set MDNAC_REPO_DIR, "
        "or in Colab clone/mount the repo under /content or Google Drive."
    )


REPO_DIR = find_repo_dir_for_import(Path.cwd())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from libs.notebook_runtime import bootstrap_notebook, materialize_notebook_training_config
from libs.core.pretrain.training_config import load_protein_training_config
from libs.core.pretrain.protein_lm.trainer import ProteinPretrainTrainer

RUNTIME = bootstrap_notebook(REPO_DIR)
REPO_DIR = Path(RUNTIME["repo_dir"])
print(f"Setup complete: {REPO_DIR}")
RUNTIME


Setup complete: C:\Users\Admin\Desktop\MDNAC


{'repo_dir': 'C:\\Users\\Admin\\Desktop\\MDNAC',
 'platform': 'Windows',
 'platform_name': 'Windows',
 'is_colab': False,
 'is_notebook': True,
 'python': '3.11.15',
 'cuda_available': True,
 'cuda_device_count': 1}

In [2]:
def activate_training_config(path):
    global CONFIG_SOURCE_PATH, CONFIG_PATH, CONFIG_NOTEBOOK_OVERRIDES, config
    source_path = Path(path).expanduser()
    if not source_path.is_absolute():
        source_path = (REPO_DIR / source_path).resolve()
    CONFIG_SOURCE_PATH = source_path
    CONFIG_PATH, CONFIG_NOTEBOOK_OVERRIDES = materialize_notebook_training_config(
        CONFIG_SOURCE_PATH,
        project_root=REPO_DIR,
    )
    config = load_protein_training_config(REPO_DIR, config_path=CONFIG_PATH)
    return config


CONFIG_SOURCE_PATH = Path(os.environ.get("MDNAC_TRAIN_CONFIG", REPO_DIR / "config" / "train.resume.yaml")).expanduser()
config = activate_training_config(CONFIG_SOURCE_PATH)
local_train_part_dir = config["paths"]["train_part_cache_dir"]
local_train_parts = sorted(local_train_part_dir.glob(config["data"]["train_part_glob"]))
train_text_path = config["paths"]["train_text_path"]
if not local_train_parts and not train_text_path.exists():
    raise FileNotFoundError(
        "Local training data not found. Expected train_part_*.txt in "
        f"{local_train_part_dir} or train.txt at {train_text_path}."
    )

config_summary = {
    "source_config_path": str(CONFIG_SOURCE_PATH),
    "effective_config_path": str(CONFIG_PATH),
    "notebook_overrides": CONFIG_NOTEBOOK_OVERRIDES,
    "mode": config["mode"],
    "optimizer_type": config["optimizer"]["type"],
    "device": config["runtime"]["device"],
    "multi_gpu_mode": config["runtime"]["multi_gpu_mode"],
    "mixed_precision": config["runtime"]["mixed_precision"],
    "checkpoint_dir": str(config["paths"]["checkpoint_dir"]),
    "resume_state_path": str(config["paths"]["resume_state_path"]),
    "train_part_cache_dir": str(config["paths"]["train_part_cache_dir"]),
    "train_part_glob": config["data"]["train_part_glob"],
    "train_text_path": str(train_text_path),
    "local_train_parts": len(local_train_parts),
    "minio_enabled": bool(config["minio"]["train_parts_prefix_uri"] or config["minio"]["train_part_uris"]),
    "num_epochs": config["training"]["num_epochs"],
    "max_steps": config["training"].get("max_steps"),
    "batch_size": config["data"]["batch_size"],
    "num_workers": config["data"]["num_workers"],
    "context_length": config["model"]["context_length"],
    "target_vram_gb": config["runtime"].get("target_vram_gb"),
    "runtime": RUNTIME,
}
config_summary


{'source_config_path': 'C:\\Users\\Admin\\Desktop\\MDNAC\\config\\train.resume.yaml',
 'effective_config_path': 'C:\\Users\\Admin\\Desktop\\MDNAC\\.notebook_runtime\\train.resume.notebook.yaml',
 'notebook_overrides': ['data.num_workers: 4 -> 0'],
 'mode': {'name': 'resume', 'resume_if_available': True},
 'optimizer_type': 'muon',
 'device': 'auto',
 'multi_gpu_mode': 'auto',
 'mixed_precision': 'fp16',
 'checkpoint_dir': 'C:\\Users\\Admin\\Desktop\\MDNAC\\data\\checkpoints\\protein_from_scratch',
 'resume_state_path': 'C:\\Users\\Admin\\Desktop\\MDNAC\\data\\checkpoints\\protein_from_scratch\\resume_state.json',
 'train_part_cache_dir': 'C:\\Users\\Admin\\Desktop\\MDNAC\\data\\cache\\protein_train_parts',
 'train_part_glob': 'train_part_*.txt',
 'train_text_path': 'C:\\Users\\Admin\\Desktop\\MDNAC\\data\\compiled\\refseq_bacteria_protein\\train.txt',
 'local_train_parts': 5,
 'minio_enabled': False,
 'num_epochs': 1,
 'max_steps': 6760,
 'batch_size': 5,
 'num_workers': 0,
 'context_l

## VRAM Check

Before training, verify that the config fits within your GPU's VRAM budget.
This cell loads the real `tokenizer_map.json`, instantiates the model, and estimates memory usage.

In [3]:
# ── VRAM Preflight Check ──────────────────────────────────────────────────────
from libs.data.training.tokenizer import SequenceTokenizer
from libs.core.pretrain.protein_lm.memory import (
    estimate_protein_pretrain_memory,
    _resolve_dtype_from_mixed_precision,
)
from libs.core.pretrain.protein_lm.support.backbone import (
    build_mdc_config_from_progen_config,
    build_progen_config,
)

# Load real tokenizer
tokenizer_map_path = config["paths"]["tokenizer_map_path"]
assert tokenizer_map_path.exists(), f"tokenizer_map.json not found: {tokenizer_map_path}"
tokenizer = SequenceTokenizer.load_map(tokenizer_map_path)
print(f"✅ Tokenizer loaded — vocab_size = {tokenizer.vocab_size}")

# Build model config with mixed precision dtype
mixed_precision = config["runtime"]["mixed_precision"]
resolved_dtype = _resolve_dtype_from_mixed_precision(mixed_precision)
model_cfg = config["model"]
progen_config = build_progen_config(
    model_cfg["progen_model_size"],
    vocab_size=tokenizer.vocab_size,
    context_length=model_cfg["context_length"],
    dtype=resolved_dtype,
)
overrides = model_cfg["progen_config_overrides"]
if overrides:
    progen_config = {**progen_config, **overrides}
model_config = build_mdc_config_from_progen_config(progen_config, dtype=resolved_dtype)

# Estimate VRAM
import torch
estimate = estimate_protein_pretrain_memory(
    model_config=model_config,
    tokenizer=tokenizer,
    batch_size=config["data"]["batch_size"],
    context_length=model_cfg["context_length"],
    optimizer_type=config["optimizer"]["type"],
    dtype=resolved_dtype,
    mixed_precision=mixed_precision,
)

runtime_cfg = config.get("runtime", {})
configured_target_vram_gb = runtime_cfg.get("target_vram_gb")

if configured_target_vram_gb is not None:
    target_budget = float(configured_target_vram_gb)
    budget_source = f"{CONFIG_PATH.name}: runtime.target_vram_gb"
elif torch.cuda.is_available():
    detected_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    target_budget = min(detected_vram_gb * 0.85, detected_vram_gb - 2.0)
    budget_source = "detected CUDA VRAM with safety margin"
else:
    raise RuntimeError("Set runtime.target_vram_gb in the training YAML before running VRAM preflight without CUDA.")

print(f"\n{'='*60}")
print(f"VRAM ESTIMATE (resolved_vocab_size={tokenizer.vocab_size})")
print(f"{'='*60}")
print(f"  Parameters:      {estimate['param_count']:,} ({estimate['param_memory_gb']:.3f} GB)")
print(f"  Gradients:       {estimate['gradient_memory_gb']:.3f} GB")
print(f"  Optimizer state: {estimate['optimizer_state_gb']:.3f} GB")
print(f"  Activations:     {estimate['activation_memory_gb']:.3f} GB")
print(f"  Logits:          {estimate['logits_memory_gb']:.3f} GB")
print(f"  TOTAL:           {estimate['total_estimate_gb']:.3f} GB")
print(f"  Budget:          {target_budget:.3f} GB ({budget_source})")
print(f"  Margin:          {target_budget - estimate['total_estimate_gb']:+.3f} GB")
if torch.cuda.is_available():
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"  GPU total VRAM:  {total_vram_gb:.3f} GB")

fits = estimate["total_estimate_gb"] <= target_budget
if fits:
    print(f"\n  ✓ Estimated to fit within configured budget ({target_budget:.1f}GB)")
else:
    print(f"\n  ✗ EXCEEDS configured budget ({target_budget:.1f}GB)!")

if not torch.cuda.is_available():
    print("  ⚠ No CUDA — this is only an estimate. Actual peak may be higher.")

✅ Tokenizer loaded — vocab_size = 256
The MDC fast path is unavailable (missing optional libraries: causal-conv1d, flash-linear-attention). Falling back to the torch implementation. The fallback uses fp32 computation (2x VRAM for activations).

VRAM ESTIMATE (resolved_vocab_size=256)
  Parameters:      280,021,632 (0.522 GB)
  Gradients:       1.043 GB
  Optimizer state: 1.045 GB
  Activations:     1.094 GB
  Logits:          0.001 GB
  TOTAL:           3.705 GB
  Budget:          14.000 GB (train.resume.notebook.yaml: runtime.target_vram_gb)
  Margin:          +10.295 GB
  GPU total VRAM:  15.928 GB

  ✓ Estimated to fit within configured budget (14.0GB)


In [4]:
USE_CONFIG = "resume"

if USE_CONFIG == "16gb":
    config = activate_training_config(REPO_DIR / "config" / "train.16gb.yaml")
    print("Using config/train.16gb.yaml")
elif USE_CONFIG == "resume":
    config = activate_training_config(REPO_DIR / "config" / "train.resume.yaml")
    print("Using config/train.resume.yaml")
elif USE_CONFIG == "current":
    print(f"Using current config: {CONFIG_PATH}")
else:
    config = activate_training_config(USE_CONFIG)
    print(f"Using custom config: {CONFIG_SOURCE_PATH}")

print(f"  effective_config={CONFIG_PATH}")
if CONFIG_NOTEBOOK_OVERRIDES:
    print(f"  notebook_overrides={CONFIG_NOTEBOOK_OVERRIDES}")
print(f"  batch_size={config['data']['batch_size']}, "
      f"context_length={config['model']['context_length']}, "
      f"grad_accum={config['training'].get('gradient_accumulation_steps', 1)}")

# Safety check: warn if estimated OOM
preflight = config.get("runtime", {}).get("preflight_vram_check", False) if isinstance(config.get("runtime"), dict) else False
if not fits and preflight:
    raise RuntimeError(
        f"Config estimated to use {estimate['total_estimate_gb']:.2f}GB "
        f"which exceeds target budget of {target_budget:.2f}GB. "
        f"Increase runtime.target_vram_gb in the YAML or reduce batch_size/context_length."
    )
elif not fits:
    print(f"WARNING: Config may exceed target VRAM budget ({target_budget:.2f}GB).")


Using config/train.resume.yaml
  effective_config=C:\Users\Admin\Desktop\MDNAC\.notebook_runtime\train.resume.notebook.yaml
  notebook_overrides=['data.num_workers: 4 -> 0']
  batch_size=5, context_length=512, grad_accum=22


In [5]:
# -- Train --------------------------------------------------------------------
trainer = ProteinPretrainTrainer(REPO_DIR, config_path=CONFIG_PATH)
result = trainer.train()
result


🚀 Training mode: resume
📥 [1/6] Loading checkpoint: checkpoint_best.pt
📦 [2/6] Restoring tokenizer from checkpoint...
✅ Tokenizer restored — vocab_size=256
🧠 [3/6] Restoring model from checkpoint...
✅ Model restored
⚙️  [4/6] Preparing runtime...
✅ Device: cuda
🔧 [5/6] Creating optimizer & restoring state...
✅ Resumed from epoch=3 step=4780 tokens=115,013,287
🔍 Running VRAM preflight check (target=14.0GB)...
✅ Preflight passed — peak=11.20GB / target=14.0GB
📂 [6/6] Building data loaders...
✅ Data loaders ready
🏋️ Resuming training loop...
📈 Epoch 4/4 | step=4780/6760 | tokens=115,013,287
  🔄 step=4790 | loss=4.3986 | tokens=115,218,762
  🔄 step=4800 | loss=3.8720 | tokens=115,426,748
  📊 eval step=4800 | train=4.1991 | val=3.7811
  💾 Saving checkpoint at step 4800...
  🔄 step=4810 | loss=4.3754 | tokens=115,629,944
  🔄 step=4820 | loss=3.5689 | tokens=115,829,668
  📊 eval step=4820 | train=4.1813 | val=3.7913
  🔄 step=4830 | loss=4.4215 | tokens=116,041,320
  🔄 step=4840 | loss=3.8393 

KeyboardInterrupt: 

In [6]:
# ── Generate sample ───────────────────────────────────────────────────────────
from libs.core.pretrain.protein_lm.core import generate_protein_text

if trainer.is_main_process and trainer.model is not None:
    sample = generate_protein_text(
        trainer.model,
        trainer.tokenizer,
        prompt="<|protein|>",
        device=trainer.runtime.device,
        max_new_tokens=80,
        context_length=int(trainer.model_config.context_length),
    )
    print(sample)
else:
    print("Sample generation is emitted on rank 0 only.")

<|protein|>MNHTTPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPPAPP


In [7]:
# ── Upload model to HuggingFace ───────────────────────────────────────────────
from huggingface_hub import HfApi, login

HF_REPO_ID = "admincybers2/MDNAC-protein-pretrain"  # ← change repo name as needed

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    print("HF_TOKEN is not set; skipping HuggingFace upload.")
else:
    login(token=hf_token)
    api = HfApi()

    checkpoint_path = result.checkpoint_path
    best_path = result.best_checkpoint_path

    files_to_upload = []
    if checkpoint_path and checkpoint_path.exists():
        files_to_upload.append((str(checkpoint_path), checkpoint_path.name))
    if best_path and best_path.exists():
        files_to_upload.append((str(best_path), best_path.name))

    tokenizer_map = config["paths"]["tokenizer_map_path"]
    if Path(tokenizer_map).exists():
        files_to_upload.append((str(tokenizer_map), "tokenizer_map.json"))

    resume_state = config["paths"]["resume_state_path"]
    if Path(resume_state).exists():
        files_to_upload.append((str(resume_state), "resume_state.json"))

    for local_path, repo_filename in files_to_upload:
        print(f"Uploading {repo_filename} ...")
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=repo_filename,
            repo_id=HF_REPO_ID,
            repo_type="model",
        )

    print(f"✅ Uploaded {len(files_to_upload)} file(s) to https://huggingface.co/{HF_REPO_ID}")

c:\Users\Admin\Desktop\MDNAC\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 